# 🩺 SepsisSense — XGBoost Training Pipeline

End-to-end pipeline that:
1. Loads all PhysioNet 2019 `.psv` files
2. Adds rolling-mean vital-signs features
3. Preprocesses (forward-fill, backward-fill, fill-0)
4. Trains an XGBoost classifier with auto-balanced `scale_pos_weight`
5. Evaluates with Accuracy, Precision, Recall, F1, ROC-AUC
6. Saves `models/sepsissense_xgb.pkl` and `models/feature_columns.pkl`

In [ ]:
import sys, os
# Make sure src/ is on the path
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, os.path.join(ROOT, 'src'))

print('Project root:', ROOT)

## Step 1 — Load Data

Loads every `.psv` file from both `training_setA` and `training_setB`.

In [ ]:
from train_model import load_data

DATA_DIR = os.path.join(ROOT, 'physionet.org', 'files', 'challenge-2019', '1.0.0', 'training')
df = load_data(DATA_DIR)
df.head()

## Step 2 — Dataset Overview

In [ ]:
import pandas as pd

print(f'Total records  : {len(df):,}')
print(f'Total patients : {df["Patient"].nunique():,}')
print(f'Columns        : {df.shape[1]}')
print(f'Sepsis rate    : {df["SepsisLabel"].mean():.3%}')
print()
df.info()

## Step 3 — Feature Engineering (Rolling Means)

Adds 6-hour rolling mean columns for the five key vital signs:
`HR`, `MAP`, `Temp`, `Resp`, `O2Sat`

In [ ]:
from train_model import add_rolling_features

df = add_rolling_features(df, vitals=['HR', 'MAP', 'Temp', 'Resp', 'O2Sat'], window=6)
new_cols = [c for c in df.columns if 'roll' in c]
print('New rolling features:', new_cols)
df[new_cols].describe()

## Step 4 — Preprocessing

Per-patient forward-fill → backward-fill → fill-0.  
Separates features `X` and target `y`.

In [ ]:
from train_model import preprocess_data

X, y = preprocess_data(df)
print('X shape:', X.shape)
print('y value counts:')
print(y.value_counts())

## Step 5 — Train / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape[0]:,} rows')
print(f'Test : {X_test.shape[0]:,} rows')

## Step 6 — Train XGBoost Model

Parameters:
- `n_estimators = 300`
- `max_depth = 5`
- `learning_rate = 0.05`
- `subsample = 0.8`
- `colsample_bytree = 0.8`
- `scale_pos_weight` — auto-calculated from class imbalance

In [ ]:
from train_model import train_model

model = train_model(X_train, y_train)
print('Training complete:', model)

## Step 7 — Evaluate & Plot Feature Importance

In [ ]:
from train_model import evaluate_model

metrics = evaluate_model(model, X_test, y_test, plot=True)

## Step 8 — Save Model and Feature Columns

In [ ]:
from train_model import save_model

save_model(model, list(X.columns))

import os
model_path = os.path.join(ROOT, 'models', 'sepsissense_xgb.pkl')
feat_path  = os.path.join(ROOT, 'models', 'feature_columns.pkl')
print('Model exists:', os.path.exists(model_path))
print('Features exists:', os.path.exists(feat_path))

## Step 9 — Verify Saved Model (Reload & Predict)

In [ ]:
import joblib

loaded_model   = joblib.load(model_path)
loaded_features = joblib.load(feat_path)

print(f'Loaded model    : {type(loaded_model).__name__}')
print(f'Feature columns : {len(loaded_features)} columns')

# Sanity-check prediction
sample = X_test.iloc[:5][loaded_features]
probs = loaded_model.predict_proba(sample)[:, 1]
print('\nSample sepsis probabilities:', probs.round(4))

---
## ✅ Pipeline Complete

| File | Location |
|------|----------|
| Trained model | `models/sepsissense_xgb.pkl` |
| Feature list  | `models/feature_columns.pkl` |
| Feature importance plot | `models/feature_importance.png` |

The model is ready to be served by `app/streamlit_app.py`.